# TLC Trip Data — Data Quality Checks

Systematic quality audit across four NYC TLC datasets:
- **Yellow** — `group3_gp.testing.yellow`
- **Green** — `group3_gp.testing.green`
- **FHV** — `group3_gp.testing.for_hire_vehicles`
- **High Volume FHV** — `group3_gp.testing.high_volume_fhv`

> **Note:** All numeric columns in these tables are stored as STRING. Queries below use `TRY_CAST` for safe type conversion.

In [0]:
from pyspark.sql import functions as F

tables = [
    "group3_gp.testing.yellow",
    "group3_gp.testing.green",
    "group3_gp.testing.for_hire_vehicles",
    "group3_gp.testing.high_volume_fhv",
]

expected_nulls = {
    "group3_gp.testing.yellow": [
        "pickup_latitude", "pickup_longitude",
        "dropoff_latitude", "dropoff_longitude",
        "pulocationid", "dolocationid",
        "improvement_surcharge", "congestion_surcharge", "airport_fee",
        "vendor_id", "pickup_datetime", "dropoff_datetime",
        "rate_code", "imp_surcharge",
        "pickup_location", "dropoff_location",
    ],
    "group3_gp.testing.green": [
        "pickup_latitude", "pickup_longitude",
        "dropoff_latitude", "dropoff_longitude",
        "pulocationid", "dolocationid",
        "ehail_fee", "congestion_surcharge", "improvement_surcharge",
    ],
    "group3_gp.testing.for_hire_vehicles": [
        "pulocationid", "sr_flag", "affiliated_base_number",
    ],
    "group3_gp.testing.high_volume_fhv": [
        "base_passenger_fare", "driver_pay", "tips", "tolls",
        "bcf", "sales_tax", "congestion_surcharge", "airport_fee",
        "shared_request_flag", "shared_match_flag", "sr_flag",
        "on_scene_datetime", "originating_base_num",
        "access_a_ride_flag", "wav_request_flag", "wav_match_flag",
    ]
}

for table in tables:
    df = spark.read.table(table)
    expected = expected_nulls.get(table, [])

    print(f"\n{'='*60}")
    print(f"Null counts for: {table}")
    print(f"{'='*60}")

    # All nulls
    display(df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df.columns
    ]))

    # Unexpected nulls only
    print("--- Unexpected nulls (genuine issues) ---")
    unexpected_cols = [c for c in df.columns if c not in expected]
    display(df.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in unexpected_cols
    ]))

In [0]:
from pyspark.sql import functions as F

df_green = spark.read.table("group3_gp.testing.green")

print(f"Row count : {df_green.count():,}")
print(f"Columns   : {len(df_green.columns)}")

# NOTE: Green uses lpep_ prefix (not tpep_ like yellow)
# NOTE: ehail_fee is almost entirely null — will be dropped in Silver
# NOTE: trip_type: 1 = Street hail, 2 = Dispatch
df_green.printSchema()

print("\n--- Sample: old-era rows (lat/long coordinates) ---")
display(
    df_green
    .filter(F.col("pickup_latitude").isNotNull())
    .select(
        "lpep_pickup_datetime", "lpep_dropoff_datetime",
        "pickup_latitude", "pickup_longitude",
        "pulocationid", "fare_amount",
        "vendorid", "trip_type", "ehail_fee", "Year"
    )
    .limit(5)
)

print("\n--- Sample: new-era rows (zone IDs) ---")
display(
    df_green
    .filter(F.col("pulocationid").isNotNull())
    .select(
        "lpep_pickup_datetime", "lpep_dropoff_datetime",
        "pulocationid", "dolocationid",
        "fare_amount", "tip_amount",
        "congestion_surcharge", "trip_type", "Year"
    )
    .limit(5)
)

# ehail_fee: confirm it is a drop candidate
total = df_green.count()
ehail_nulls = df_green.filter(F.col("ehail_fee").isNull()).count()
print(f"\nehail_fee: {ehail_nulls:,} nulls out of {total:,} rows "
      f"({100 * ehail_nulls / total:.1f}%) — DROP this column in Silver")

print("\n--- Summary statistics ---")
df_green.describe().show()

In [0]:
from pyspark.sql import functions as F

df_fhv = spark.read.table("group3_gp.testing.for_hire_vehicles")

print(f"Row count : {df_fhv.count():,}")
print(f"Columns   : {len(df_fhv.columns)}")

# NOTE: Only 8 columns — very sparse by design
# No fare or distance data — TLC only required base number, datetime, location
# dropoff_datetime and dropoff location only added from 2017 onwards
# affiliated_base_number links to the base that received the original request
df_fhv.printSchema()
display(df_fhv.limit(20))

# Check for dirty strings in affiliated_base_number
# Valid base numbers follow pattern: B followed by 5 digits e.g. B02395
print("\n--- Dirty values in affiliated_base_number ---")
display(
    df_fhv
    .filter(
        F.col("affiliated_base_number").isNotNull() &
        ~F.col("affiliated_base_number").rlike("^B[0-9]{5}$")
    )
    .groupBy("affiliated_base_number")
    .count()
    .orderBy(F.col("count").desc())
)

# sr_flag: null = non-shared, 1 = shared ride
print("\n--- sr_flag distribution (null = non-shared, 1 = shared) ---")
display(df_fhv.groupBy("sr_flag").count().orderBy("sr_flag"))

print("\n--- Summary statistics ---")
df_fhv.describe().show()

In [0]:
from pyspark.sql import functions as F

df_hv = spark.read.table("group3_gp.testing.high_volume_fhv")

print(f"Row count : {df_hv.count():,}")
print(f"Columns   : {len(df_hv.columns)}")

# NOTE: hvfhs_license_num identifies the company:
#   HV0002 = Juno | HV0003 = Uber | HV0004 = Via | HV0005 = Lyft
# NOTE: Fare columns only populated from ~2021 onwards
#   Nulls in earlier years are EXPECTED — do not treat as errors
# NOTE: on_scene_datetime only populated for WAV (wheelchair-accessible) trips
# NOTE: Lyft (HV0005) overcounts shared rides —
#   also flags requested-but-unmatched rides as shared
df_hv.printSchema()
display(df_hv.limit(20))

# Company breakdown using hvfhs_license_num from data dictionary
print("\n--- Ride counts by company ---")
display(
    df_hv
    .withColumn("company",
        F.when(F.col("hvfhs_license_num") == "HV0002", "Juno")
         .when(F.col("hvfhs_license_num") == "HV0003", "Uber")
         .when(F.col("hvfhs_license_num") == "HV0004", "Via")
         .when(F.col("hvfhs_license_num") == "HV0005", "Lyft")
         .otherwise("Unknown")
    )
    .groupBy("company", "hvfhs_license_num")
    .count()
    .orderBy(F.col("count").desc())
)

# Fare data availability by year — confirms nulls expected pre-2021
print("\n--- Fare data availability by year (nulls pre-2021 are EXPECTED) ---")
display(
    df_hv
    .groupBy("Year")
    .agg(
        F.count("*").alias("total_rows"),
        F.count("base_passenger_fare").alias("fare_populated"),
        F.count("driver_pay").alias("driver_pay_populated"),
        F.round(
            F.count("base_passenger_fare") / F.count("*") * 100, 1
        ).alias("fare_coverage_%")
    )
    .orderBy("Year")
)

# WAV (wheelchair-accessible vehicle) trip breakdown
print("\n--- Wheelchair-accessible vehicle (WAV) trips ---")
display(
    df_hv
    .groupBy("wav_request_flag", "wav_match_flag")
    .count()
    .orderBy(F.col("count").desc())
)

# Shared ride breakdown
# NOTE: Lyft overcounts — sr_flag=1 includes requested-but-unmatched shared rides
print("\n--- Shared rides by company (NOTE: Lyft figures are overstated) ---")
display(
    df_hv
    .withColumn("company",
        F.when(F.col("hvfhs_license_num") == "HV0002", "Juno")
         .when(F.col("hvfhs_license_num") == "HV0003", "Uber")
         .when(F.col("hvfhs_license_num") == "HV0004", "Via")
         .when(F.col("hvfhs_license_num") == "HV0005", "Lyft")
         .otherwise("Unknown")
    )
    .groupBy("company", "shared_request_flag", "shared_match_flag")
    .count()
    .orderBy("company", F.col("count").desc())
)

print("\n--- Summary statistics ---")
df_hv.describe().show()

In [0]:
from pyspark.sql import functions as F

# Columns that are EXPECTED to be null — based on data dictionaries
# These are NOT data quality issues and should not be flagged as errors
expected_nulls = {
    "group3_gp.testing.yellow": [
        # Pre-2017 rows use lat/long, post-2017 rows use zone IDs — mutually exclusive
        "pickup_latitude", "pickup_longitude",
        "dropoff_latitude", "dropoff_longitude",
        "pulocationid", "dolocationid",
        # Surcharges introduced at different years
        "improvement_surcharge",   # started 2015
        "congestion_surcharge",    # started 2019
        "airport_fee",             # started 2022
        # Duplicate column name variants — one era always null
        "vendor_id", "pickup_datetime", "dropoff_datetime",
        "rate_code", "imp_surcharge",
        # GeoJSON structs — not useful, will be dropped
        "pickup_location", "dropoff_location",
    ],
    "group3_gp.testing.green": [
        "pickup_latitude", "pickup_longitude",
        "dropoff_latitude", "dropoff_longitude",
        "pulocationid", "dolocationid",
        "ehail_fee",             # never widely used — drop in Silver
        "congestion_surcharge",
        "improvement_surcharge",
    ],
    "group3_gp.testing.for_hire_vehicles": [
        "pulocationid",          # pickup location not always recorded
        "sr_flag",               # null = non-shared ride (expected for most rows)
        "affiliated_base_number" # not always provided
    ],
    "group3_gp.testing.high_volume_fhv": [
        # Fare columns only available from ~2021 — nulls pre-2021 are expected
        "base_passenger_fare", "driver_pay", "tips", "tolls",
        "bcf", "sales_tax", "congestion_surcharge", "airport_fee",
        "shared_request_flag", "shared_match_flag",
        "sr_flag",               # null = non-shared ride
        "on_scene_datetime",     # only for WAV/accessible vehicle trips
        "originating_base_num",
        "access_a_ride_flag",
        "wav_request_flag", "wav_match_flag",
    ]
}

tables = [
    "group3_gp.testing.yellow",
    "group3_gp.testing.green",
    "group3_gp.testing.for_hire_vehicles",
    "group3_gp.testing.high_volume_fhv",
]

for table in tables:
    df = spark.read.table(table)
    expected = expected_nulls.get(table, [])

    print(f"\n{'='*60}")
    print(f"Null counts for: {table}")
    print(f"{'='*60}")

    # Full null count for every column
    display(
        df.select([
            F.count(F.when(F.col(c).isNull(), c)).alias(c)
            for c in df.columns
        ])
    )

    # Highlight only UNEXPECTED nulls — columns that should always be populated
    print("\n--- Unexpected nulls only (genuine data quality issues) ---")
    unexpected_cols = [c for c in df.columns if c not in expected]
    display(
        df.select([
            F.count(F.when(F.col(c).isNull(), c)).alias(c)
            for c in unexpected_cols
        ])
    )

In [0]:
# ============================================================
# Bad data checks for: group3_gp.testing.yellow
# Rewritten with SQL + TRY_CAST to safely handle all-STRING columns
# ============================================================

# --- Summary counts (single-pass scan) ---
print("--- Yellow taxi: data quality summary ---")
display(spark.sql("""
  WITH base AS (
    SELECT *,
      COALESCE(TRY_CAST(tpep_pickup_datetime AS TIMESTAMP),
               TRY_CAST(pickup_datetime AS TIMESTAMP))      AS pickup_dt,
      COALESCE(TRY_CAST(tpep_dropoff_datetime AS TIMESTAMP),
               TRY_CAST(dropoff_datetime AS TIMESTAMP))     AS dropoff_dt,
      CAST(TRY_CAST(COALESCE(ratecodeid, rate_code) AS DOUBLE) AS INT) AS ratecode_clean
    FROM group3_gp.testing.yellow
  )
  SELECT
    COUNT(*)                                                              AS total_rows,
    -- Exclude payment_type 3 (No charge) and 6 (Voided) from bad-fare count
    COUNT_IF(TRY_CAST(fare_amount AS DOUBLE) <= 0
             AND payment_type NOT IN ('3', '6'))                          AS bad_fares,
    COUNT_IF(TRY_CAST(fare_amount AS DOUBLE) <= 0
             AND payment_type IN ('3', '6'))                              AS legit_zero_fares,
    COUNT_IF(TRY_CAST(trip_distance AS DOUBLE) <= 0)                      AS bad_distances,
    COUNT_IF(dropoff_dt <= pickup_dt)                                     AS bad_timestamps,
    COUNT_IF(ratecode_clean = 99)                                         AS invalid_ratecodes,
    COUNT_IF(store_and_fwd_flag = 'Y')                                    AS stored_offline,
    COUNT_IF(TRY_CAST(passenger_count AS DOUBLE) NOT BETWEEN 1 AND 8)     AS invalid_passenger_count
  FROM base
"""))

# --- Payment type breakdown ---
print("\n--- Payment type breakdown ---")
display(spark.sql("""
  SELECT
    CASE payment_type
      WHEN '0' THEN 'Flex fare'
      WHEN '1' THEN 'Credit card'
      WHEN '2' THEN 'Cash'
      WHEN '3' THEN 'No charge'
      WHEN '4' THEN 'Dispute'
      WHEN '5' THEN 'Unknown'
      WHEN '6' THEN 'Voided trip'
      ELSE 'Unrecognised'
    END AS payment_description,
    COUNT(*) AS cnt
  FROM group3_gp.testing.yellow
  GROUP BY payment_type
  ORDER BY cnt DESC
"""))

# --- Rate code breakdown ---
print("\n--- Rate code breakdown ---")
display(spark.sql("""
  SELECT
    CASE CAST(TRY_CAST(COALESCE(ratecodeid, rate_code) AS DOUBLE) AS INT)
      WHEN 1  THEN 'Standard rate'
      WHEN 2  THEN 'JFK'
      WHEN 3  THEN 'Newark'
      WHEN 4  THEN 'Nassau or Westchester'
      WHEN 5  THEN 'Negotiated fare'
      WHEN 6  THEN 'Group ride'
      WHEN 99 THEN 'Unknown / invalid'
      ELSE 'Unrecognised'
    END AS rate_description,
    COUNT(*) AS cnt
  FROM group3_gp.testing.yellow
  GROUP BY 1
  ORDER BY cnt DESC
"""))

# --- Year distribution ---
print("\n--- Year distribution ---")
display(spark.sql("""
  SELECT
    YEAR(COALESCE(TRY_CAST(tpep_pickup_datetime AS TIMESTAMP),
                  TRY_CAST(pickup_datetime AS TIMESTAMP))) AS pickup_year,
    COUNT(*) AS cnt
  FROM group3_gp.testing.yellow
  GROUP BY 1
  ORDER BY pickup_year
"""))

# --- Duplicates (expensive — full shuffle) ---
total    = spark.sql("SELECT COUNT(*) AS c FROM group3_gp.testing.yellow").first()["c"]
distinct = spark.sql("SELECT COUNT(*) AS c FROM (SELECT DISTINCT * FROM group3_gp.testing.yellow)").first()["c"]
print(f"\nDuplicates: {total - distinct:,}")

In [0]:
# ============================================================
# Bad data checks for: group3_gp.testing.green
# ============================================================

# --- Summary counts ---
print("--- Green taxi: data quality summary ---")
display(spark.sql("""
  WITH base AS (
    SELECT *,
      TRY_CAST(lpep_pickup_datetime AS TIMESTAMP)   AS pickup_dt,
      TRY_CAST(lpep_dropoff_datetime AS TIMESTAMP)  AS dropoff_dt
    FROM group3_gp.testing.green
  )
  SELECT
    COUNT(*)                                                              AS total_rows,
    COUNT_IF(TRY_CAST(fare_amount AS DOUBLE) <= 0
             AND payment_type NOT IN ('3', '6'))                          AS bad_fares,
    COUNT_IF(TRY_CAST(trip_distance AS DOUBLE) <= 0)                      AS bad_distances,
    COUNT_IF(dropoff_dt <= pickup_dt)                                     AS bad_timestamps,
    COUNT_IF(TRY_CAST(passenger_count AS DOUBLE) NOT BETWEEN 1 AND 8)     AS invalid_passenger_count,
    COUNT_IF(ehail_fee IS NULL)                                           AS ehail_fee_nulls,
    ROUND(COUNT_IF(ehail_fee IS NULL) * 100.0 / COUNT(*), 1)              AS ehail_fee_null_pct
  FROM base
"""))

# --- Trip type breakdown (1 = Street hail, 2 = Dispatch) ---
print("\n--- Trip type breakdown ---")
display(spark.sql("""
  SELECT
    CASE trip_type
      WHEN '1' THEN 'Street hail'
      WHEN '2' THEN 'Dispatch'
      ELSE 'Unknown'
    END AS trip_type_desc,
    COUNT(*) AS cnt
  FROM group3_gp.testing.green
  GROUP BY trip_type
  ORDER BY cnt DESC
"""))

# --- Year distribution ---
print("\n--- Year distribution ---")
display(spark.sql("""
  SELECT
    YEAR(TRY_CAST(lpep_pickup_datetime AS TIMESTAMP)) AS pickup_year,
    COUNT(*) AS cnt
  FROM group3_gp.testing.green
  GROUP BY 1
  ORDER BY pickup_year
"""))

# --- Duplicates ---
total    = spark.sql("SELECT COUNT(*) AS c FROM group3_gp.testing.green").first()["c"]
distinct = spark.sql("SELECT COUNT(*) AS c FROM (SELECT DISTINCT * FROM group3_gp.testing.green)").first()["c"]
print(f"\nDuplicates: {total - distinct:,}")

In [0]:
# ============================================================
# Bad data checks for: group3_gp.testing.for_hire_vehicles
# No fare or distance data — only timestamps, base numbers, locations
# ============================================================

# --- Summary counts ---
print("--- FHV: data quality summary ---")
display(spark.sql("""
  SELECT
    COUNT(*)                                                              AS total_rows,
    COUNT_IF(TRY_CAST(dropoff_datetime AS TIMESTAMP)
             <= TRY_CAST(pickup_datetime AS TIMESTAMP))                   AS bad_timestamps,
    COUNT_IF(affiliated_base_number IS NOT NULL
             AND affiliated_base_number NOT RLIKE '^B[0-9]{5}$')          AS dirty_base_numbers
  FROM group3_gp.testing.for_hire_vehicles
"""))

# --- Dirty affiliated_base_number samples ---
print("\n--- Dirty affiliated_base_number samples ---")
display(spark.sql("""
  SELECT affiliated_base_number, COUNT(*) AS cnt
  FROM group3_gp.testing.for_hire_vehicles
  WHERE affiliated_base_number IS NOT NULL
    AND affiliated_base_number NOT RLIKE '^B[0-9]{5}$'
  GROUP BY affiliated_base_number
  ORDER BY cnt DESC
"""))

# --- sr_flag distribution (null = non-shared, 1 = shared) ---
print("\n--- sr_flag distribution ---")
display(spark.sql("""
  SELECT sr_flag, COUNT(*) AS cnt
  FROM group3_gp.testing.for_hire_vehicles
  GROUP BY sr_flag
  ORDER BY sr_flag
"""))

# --- Year distribution ---
print("\n--- Year distribution ---")
display(spark.sql("""
  SELECT Year, COUNT(*) AS cnt
  FROM group3_gp.testing.for_hire_vehicles
  GROUP BY Year
  ORDER BY Year
"""))

# --- Duplicates ---
total    = spark.sql("SELECT COUNT(*) AS c FROM group3_gp.testing.for_hire_vehicles").first()["c"]
distinct = spark.sql("SELECT COUNT(*) AS c FROM (SELECT DISTINCT * FROM group3_gp.testing.for_hire_vehicles)").first()["c"]
print(f"\nDuplicates: {total - distinct:,}")

In [0]:
# ============================================================
# Bad data checks for: group3_gp.testing.high_volume_fhv
# Fare columns only available from ~2021 — nulls pre-2021 are EXPECTED
# Lyft (HV0005) overcounts shared rides
# ============================================================

# --- Summary counts ---
print("--- High Volume FHV: data quality summary ---")
display(spark.sql("""
  SELECT
    COUNT(*)                                                              AS total_rows,
    COUNT_IF(TRY_CAST(dropoff_datetime AS TIMESTAMP)
             <= TRY_CAST(pickup_datetime AS TIMESTAMP))                   AS bad_timestamps,
    COUNT_IF(TRY_CAST(trip_miles AS DOUBLE) <= 0)                         AS bad_distances,
    COUNT_IF(base_passenger_fare IS NOT NULL
             AND TRY_CAST(base_passenger_fare AS DOUBLE) <= 0)            AS bad_fares,
    COUNT_IF(driver_pay IS NOT NULL
             AND TRY_CAST(driver_pay AS DOUBLE) <= 0)                     AS bad_driver_pay
  FROM group3_gp.testing.high_volume_fhv
"""))

# --- Fare availability by year (nulls pre-2021 are EXPECTED) ---
print("\n--- Fare availability by year ---")
display(spark.sql("""
  SELECT
    Year,
    COUNT(*) AS total_rows,
    COUNT(base_passenger_fare) AS fare_populated,
    COUNT(driver_pay) AS driver_pay_populated,
    ROUND(COUNT(base_passenger_fare) * 100.0 / COUNT(*), 1) AS `fare_coverage_%`
  FROM group3_gp.testing.high_volume_fhv
  GROUP BY Year
  ORDER BY Year
"""))

# --- Trips by company ---
# HV0002 = Juno | HV0003 = Uber | HV0004 = Via | HV0005 = Lyft
print("\n--- Trips by company ---")
display(spark.sql("""
  SELECT
    CASE hvfhs_license_num
      WHEN 'HV0002' THEN 'Juno'
      WHEN 'HV0003' THEN 'Uber'
      WHEN 'HV0004' THEN 'Via'
      WHEN 'HV0005' THEN 'Lyft'
      ELSE 'Unknown'
    END AS company,
    COUNT(*) AS cnt
  FROM group3_gp.testing.high_volume_fhv
  GROUP BY hvfhs_license_num
  ORDER BY cnt DESC
"""))

# --- Shared rides by company (NOTE: Lyft figures are overstated) ---
print("\n--- Shared rides by company ---")
display(spark.sql("""
  SELECT
    CASE hvfhs_license_num
      WHEN 'HV0002' THEN 'Juno'
      WHEN 'HV0003' THEN 'Uber'
      WHEN 'HV0004' THEN 'Via'
      WHEN 'HV0005' THEN 'Lyft'
      ELSE 'Unknown'
    END AS company,
    shared_request_flag, shared_match_flag,
    COUNT(*) AS cnt
  FROM group3_gp.testing.high_volume_fhv
  GROUP BY hvfhs_license_num, shared_request_flag, shared_match_flag
  ORDER BY 1, cnt DESC
"""))

# --- WAV (wheelchair-accessible vehicle) trips ---
print("\n--- WAV trips ---")
display(spark.sql("""
  SELECT wav_request_flag, wav_match_flag, COUNT(*) AS cnt
  FROM group3_gp.testing.high_volume_fhv
  GROUP BY wav_request_flag, wav_match_flag
  ORDER BY cnt DESC
"""))

# --- Year distribution ---
print("\n--- Year distribution ---")
display(spark.sql("""
  SELECT Year, COUNT(*) AS cnt
  FROM group3_gp.testing.high_volume_fhv
  GROUP BY Year
  ORDER BY Year
"""))

# --- Duplicates (expensive — full shuffle) ---
total    = spark.sql("SELECT COUNT(*) AS c FROM group3_gp.testing.high_volume_fhv").first()["c"]
distinct = spark.sql("SELECT COUNT(*) AS c FROM (SELECT DISTINCT * FROM group3_gp.testing.high_volume_fhv)").first()["c"]
print(f"\nDuplicates: {total - distinct:,}")